# GO Enrichment Analysis Overview

This notebook parses a pre-filtered Gene Ontology (GO) enrichment result file (TSV format) and produces a series of plots and tables to help interpret the data. The result file is assumed to already contain gene sets within the desired size limits.

The file is expected to have the following columns:

- **term**: GO identifier
- **name**: GO term name
- **size**: number of measured genes associated with the term
- **is_true**: boolean flag indicating whether the term is a simulated enriched GO entry
- **noverlap**: number of significant genes (SGs) among the measured ones associated with the term
- **hg_pval**: p-value from the hypergeometric test
- **hg_fdr**: Benjamini-Hochberg corrected hypergeometric p-value
- **fej_pval**: p-value from Fischer’s exact test using jack-knifing
- **fej_fdr**: Benjamini-Hochberg corrected Fischer’s p-value
- **ks_stat**: KS test statistic (enrichment score)
- **ks_pval**: p-value from the KS test
- **ks_fdr**: Benjamini-Hochberg corrected KS p-value
- **shortest_path_to_a_true**: if the term is not a true entry, the shortest path in the GO DAG to a true entry

Optionally, if a column named `sg_genes` is available (with gene IDs separated by a delimiter such as `;`), additional analysis on significant gene occurrences will be performed.

Let’s begin by loading the data and generating the plots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Use a clean and modern style for plots
sns.set(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

# Path to the result file (adjust this path as needed)
result_file = "../../../../data/GOEnrichment/runs/ensembl_BP_out.tsv"  # e.g., "simul_exp_go_bp_ensembl_min50_max500.enrich.out"

# Read the TSV file into a DataFrame. It is assumed the file has a header.
df = pd.read_csv(result_file, sep="\t")

# Show the first few lines to verify the data
display(df.head())

## 1. Distribution of Significant Genes (SGs)

Here we plot a histogram showing the distribution of the number of significant genes (the `noverlap` column) across all gene sets.

In [ ]:
plt.figure()
sns.histplot(df["noverlap"], bins=30, color="steelblue")
plt.title("Distribution of Significant Genes in All Gene Sets")
plt.xlabel("Number of Significant Genes (noverlap)")
plt.ylabel("Count")
plt.show()

## 2. Distribution of Enrichment Scores and P-values

The next section generates histograms for several enrichment metrics:

- **KS Statistic** (`ks_stat`)
- **Hypergeometric p-value** (`hg_pval`)
- **Fischer’s Exact Test p-value** (`fej_pval`)
- **KS Test p-value** (`ks_pval`)

These plots help visualize the overall distribution of the enrichment scores and statistical significance across gene sets.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# KS Statistic
sns.histplot(df["ks_stat"], bins=30, ax=axes[0, 0], color="purple")
axes[0, 0].set_title("Distribution of KS Statistic")
axes[0, 0].set_xlabel("KS Statistic")

# Hypergeometric p-value
sns.histplot(df["hg_pval"], bins=30, ax=axes[0, 1], color="tomato")
axes[0, 1].set_title("Distribution of Hypergeometric p-values")
axes[0, 1].set_xlabel("Hypergeometric p-value")

# Fischer's Exact Test p-value
sns.histplot(df["fej_pval"], bins=30, ax=axes[1, 0], color="orange")
axes[1, 0].set_title("Distribution of Fischer's Exact Test p-values")
axes[1, 0].set_xlabel("Fischer's p-value")

# KS Test p-value
sns.histplot(df["ks_pval"], bins=30, ax=axes[1, 1], color="teal")
axes[1, 1].set_title("Distribution of KS Test p-values")
axes[1, 1].set_xlabel("KS Test p-value")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Hypergeometric FDR
sns.histplot(df["hg_fdr"], bins=30, ax=axes[0], color="tomato")
axes[0].set_title("Distribution of Hypergeometric FDR")
axes[0].set_xlabel("Hypergeometric FDR")

# Fischer's Exact Test FDR
sns.histplot(df["fej_fdr"], bins=30, ax=axes[1], color="orange")
axes[1].set_title("Distribution of Fischer's FDR")
axes[1].set_xlabel("Fischer's FDR")

# KS Test FDR
sns.histplot(df["ks_fdr"], bins=30, ax=axes[2], color="teal")
axes[2].set_title("Distribution of KS FDR")
axes[2].set_xlabel("KS FDR")

plt.tight_layout()
plt.show()


## 3. Scatter Plots: Enrichment Metrics vs. Gene Set Size

### A. KS Statistic vs. Gene Set Size

This plot shows how the enrichment score (KS statistic) relates to the number of genes in each set.

In [ ]:
plt.figure()
sns.scatterplot(data=df, x="size", y="ks_stat", color="navy", alpha=0.7)
plt.title("Scatter Plot: KS Statistic vs Gene Set Size")
plt.xlabel("Gene Set Size")
plt.ylabel("KS Statistic")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 18))

# Scatter: Gene Set Size vs. Hypergeometric p-value
sns.scatterplot(data=df, x="size", y="hg_pval", ax=axes[0, 0], color="tomato", alpha=0.7)
axes[0, 0].set_title("Gene Set Size vs. Hypergeometric p-value")
axes[0, 0].set_xlabel("Gene Set Size")
axes[0, 0].set_ylabel("Hypergeometric p-value")

# Scatter: Gene Set Size vs. Hypergeometric FDR
sns.scatterplot(data=df, x="size", y="hg_fdr", ax=axes[0, 1], color="tomato", alpha=0.7)
axes[0, 1].set_title("Gene Set Size vs. Hypergeometric FDR")
axes[0, 1].set_xlabel("Gene Set Size")
axes[0, 1].set_ylabel("Hypergeometric FDR")

# Scatter: Gene Set Size vs. Fischer's p-value
sns.scatterplot(data=df, x="size", y="fej_pval", ax=axes[1, 0], color="orange", alpha=0.7)
axes[1, 0].set_title("Gene Set Size vs. Fischer's p-value")
axes[1, 0].set_xlabel("Gene Set Size")
axes[1, 0].set_ylabel("Fischer's p-value")

# Scatter: Gene Set Size vs. Fischer's FDR
sns.scatterplot(data=df, x="size", y="fej_fdr", ax=axes[1, 1], color="orange", alpha=0.7)
axes[1, 1].set_title("Gene Set Size vs. Fischer's FDR")
axes[1, 1].set_xlabel("Gene Set Size")
axes[1, 1].set_ylabel("Fischer's FDR")

# Scatter: Gene Set Size vs. KS p-value
sns.scatterplot(data=df, x="size", y="ks_pval", ax=axes[2, 0], color="teal", alpha=0.7)
axes[2, 0].set_title("Gene Set Size vs. KS p-value")
axes[2, 0].set_xlabel("Gene Set Size")
axes[2, 0].set_ylabel("KS p-value")

# Scatter: Gene Set Size vs. KS FDR
sns.scatterplot(data=df, x="size", y="ks_fdr", ax=axes[2, 1], color="teal", alpha=0.7)
axes[2, 1].set_title("Gene Set Size vs. KS FDR")
axes[2, 1].set_xlabel("Gene Set Size")
axes[2, 1].set_ylabel("KS FDR")

plt.tight_layout()
plt.show()

### B. Hypergeometric p-value vs. Gene Set Size

This plot explores the relationship between gene set size and the hypergeometric p-value.

In [ ]:
plt.figure()
sns.scatterplot(data=df, x="size", y="hg_pval", color="darkred", alpha=0.7)
plt.title("Scatter Plot: Hypergeometric p-value vs Gene Set Size")
plt.xlabel("Gene Set Size")
plt.ylabel("Hypergeometric p-value")
plt.show()

## 4. Sorted Tables of Gene Sets

Below are examples of sorted tables (displaying the top 10 entries) according to different criteria.

In [ ]:
# Sorted by number of significant genes (noverlap)
df_sorted_noverlap = df.sort_values(by="noverlap", ascending=False)
print("Top 10 Gene Sets Sorted by Number of Significant Genes (noverlap):")
display(df_sorted_noverlap[["term", "name", "noverlap"]].head(10))

# Sorted by KS Statistic (enrichment score)
df_sorted_ks = df.sort_values(by="ks_stat", ascending=False)
print("Top 10 Gene Sets Sorted by KS Statistic:")
display(df_sorted_ks[["term", "name", "ks_stat"]].head(10))

# Sorted by Hypergeometric p-value (lowest p-values first)
df_sorted_hg = df.sort_values(by="hg_pval", ascending=True)
print("Top 10 Gene Sets Sorted by Hypergeometric p-value:")
display(df_sorted_hg[["term", "name", "hg_pval"]].head(10))

## 5. Analysis of Significant Gene Occurrence Across Gene Sets (Optional)

If the result file includes a column named `sg_genes` (with gene IDs separated by a delimiter such as `;`), the following cell will analyze how often each significant gene appears across the gene sets. If the column is not present, this section is skipped.

In [ ]:
if "sg_genes" in df.columns:
    # Split the gene lists and count occurrences
    sg_series = df["sg_genes"].dropna().str.split(";").explode()
    sg_counts = sg_series.value_counts()

    # Plot the distribution of the number of gene sets per significant gene
    plt.figure()
    sns.histplot(sg_counts, bins=30, color="mediumseagreen")
    plt.title("Distribution of Number of Gene Sets per Significant Gene")
    plt.xlabel("Number of Sets a Gene Appears In")
    plt.ylabel("Count of Genes")
    plt.show()

    # Report the number of genes that appear uniquely (only in one gene set)
    unique_sg = sg_counts[sg_counts == 1]
    print(f"Number of unique significant genes (appearing in only one set): {len(unique_sg)}")
else:
    print("Column 'sg_genes' not found in the result file; skipping analysis of significant gene occurrence.")

In [ ]:


# Path to the analysis file (update the filename/path as needed)
analysis_file = "../../../../data/GOEnrichment/analysis-out-ensembl.tsv"

# Dictionary to hold the distributions; keys are the header lines, values are lists of numbers.
distributions = {}

# Read the file and parse the distributions.
with open(analysis_file, "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    line = lines[i].strip()
    # Look for a line that starts with "Distribution of"
    if line.startswith("Distribution of"):
        header = line  # e.g., "Distribution of Gene Set Sizes (All):"
        # Advance to the next non-empty line, which should contain the comma-separated numbers.
        i += 1
        while i < len(lines) and lines[i].strip() == "":
            i += 1
        if i < len(lines):
            csv_line = lines[i].strip()
            try:
                # Parse each value as an int.
                values = [int(x.strip()) for x in csv_line.split(",") if x.strip() != ""]
            except Exception as e:
                print(f"Error parsing line: {csv_line} with error {e}")
                values = []
            distributions[header] = values
    i += 1
# Now, for each distribution, plot a histogram.
for header, values in distributions.items():
    if len(values) == 0:
        continue  # Skip if no data
    plt.figure(figsize=(8, 6))
    plt.hist(values, bins='auto', color="skyblue", edgecolor="black", alpha=0.75)

    # Set x-axis label based on the header content.
    if "Gene Set Sizes" in header:
        xlabel = "Gene Set Size"
    elif "Leaf Path Lengths" in header:
        xlabel = "Path Length"
    elif "Set Differences" in header:
        xlabel = "Set Difference (Parent - Child)"
    else:
        xlabel = "Value"

    plt.title(header, fontsize=16)
    plt.xlabel(xlabel, fontsize=14)
    plt.ylabel("Frequency", fontsize=14)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# %% [code]
import matplotlib.pyplot as plt

# Path to the analysis file (update the filename/path as needed)
analysis_file = "../../../../data/GOEnrichment/analysis-out-ensembl.tsv"

# Dictionary to hold the distributions.
# Keys are the header lines; values are lists of integers.
distributions = {}

# Read the file and parse the distributions.
with open(analysis_file, "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    line = lines[i].strip()
    # Look for a line that starts with "Distribution of"
    if line.startswith("Distribution of"):
        header = line  # e.g., "Distribution of Gene Set Sizes (All):"
        # Advance to the next non-empty line, which should contain the comma-separated numbers.
        i += 1
        while i < len(lines) and lines[i].strip() == "":
            i += 1
        if i < len(lines):
            csv_line = lines[i].strip()
            try:
                # Parse each value as an int.
                values = [int(x.strip()) for x in csv_line.split(",") if x.strip() != ""]
            except Exception as e:
                print(f"Error parsing line: {csv_line} with error {e}")
                values = []
            distributions[header] = values
    i += 1

# Now, for each distribution, plot appropriately.
for header, values in distributions.items():
    if len(values) == 0:
        continue  # Skip if no data

    # Special handling for leaf path lengths: use a bar plot.
    if "Leaf Path Lengths" in header:
        # Compute frequency counts for each unique value.
        unique_vals = sorted(set(values))
        freq = [values.count(x) for x in unique_vals]
        plt.figure(figsize=(8, 6))
        plt.bar(unique_vals, freq, color="skyblue", edgecolor="black", alpha=0.75)
        plt.title(header, fontsize=16)
        plt.xlabel("Path Length", fontsize=14)
        plt.ylabel("Frequency", fontsize=14)
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.show()
    else:
        # For gene set sizes and set differences, use a histogram with a log y-axis.
        plt.figure(figsize=(8, 6))
        if "Gene Set Sizes" in header or "Set Differences" in header:
            plt.hist(values, bins="auto", color="skyblue", edgecolor="black", alpha=0.75, log=True)
        else:
            plt.hist(values, bins="auto", color="skyblue", edgecolor="black", alpha=0.75)

        # Set x-axis label based on the header content.
        if "Gene Set Sizes" in header:
            xlabel = "Gene Set Size"
        elif "Set Differences" in header:
            xlabel = "Set Difference (Parent - Child)"
        else:
            xlabel = "Value"

        plt.title(header, fontsize=16)
        plt.xlabel(xlabel, fontsize=14)
        plt.ylabel("Frequency", fontsize=14)
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.show()

In [ ]:


# --- Part 1: Parse Analysis Files ---
def parse_analysis_file(file_path):
    """
    Parses an analysis file and returns a dictionary mapping header lines (e.g.,
    "Distribution of Gene Set Sizes (All):") to lists of integers.
    """
    distributions = {}
    with open(file_path, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith("Distribution of"):
            header = line  # For example: "Distribution of Gene Set Sizes (All):"
            i += 1
            while i < len(lines) and lines[i].strip() == "":
                i += 1
            if i < len(lines):
                csv_line = lines[i].strip()
                try:
                    values = [int(x.strip()) for x in csv_line.split(",") if x.strip() != ""]
                except Exception as e:
                    print(f"Error parsing line: {csv_line} with error {e}")
                    values = []
                distributions[header] = values
        i += 1
    return distributions


# Define file paths and labels for each ontology.
files = {
    "Biological Process": "../../../../data/GOEnrichment/Runs/analysis-out-ensembl-bp.tsv",
    "Cellular Component": "../../../../data/GOEnrichment/Runs/analysis-out-ensembl-cc.tsv",
    "Molecular Function": "../../../../data/GOEnrichment/Runs/analysis-out-ensembl-mf.tsv"
}

# Parse each file into a dictionary keyed by ontology.
all_distributions = {}
ontologies = list(files.keys())
for ontology, file_path in files.items():
    all_distributions[ontology] = parse_analysis_file(file_path)

# -------------------------------
# 1. Cumulative Distribution: Gene Set Sizes (All) (Absolute Count) with symlog x-scale
# -------------------------------
cat1 = "Distribution of Gene Set Sizes (All):"
plt.figure(figsize=(10, 7))
max_val1 = 0
for ontology in ontologies:
    if cat1 in all_distributions[ontology]:
        data = np.array(all_distributions[ontology][cat1], dtype=int)
        sorted_data = np.sort(data)
        cum_count = np.arange(1, len(sorted_data) + 1)  # absolute cumulative count
        max_val1 = max(max_val1, sorted_data[-1])
        plt.step(sorted_data, cum_count, where="post", label=ontology)

plt.xscale('symlog', linthresh=1)
plt.xlim(1, max_val1 * 1.05)
plt.title("Cumulative Distribution: Gene Set Sizes (All) (Absolute Count)", fontsize=16)
plt.xlabel("Gene Set Size (symlog scale)", fontsize=14)
plt.ylabel("Cumulative Count", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("plots/cumulative_distribution_gene_set_sizes_all_absolute.png")
plt.show()

# -------------------------------
# 1. Cumulative Distribution: Gene Set Sizes (All) (Relative Proportion) (symlog x-scale)
# -------------------------------
plt.figure(figsize=(10, 7))
max_val1 = 0
for ontology in ontologies:
    if cat1 in all_distributions[ontology]:
        data = np.array(all_distributions[ontology][cat1], dtype=int)
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        max_val1 = max(max_val1, sorted_data[-1])
        plt.step(sorted_data, cdf, where="post", label=ontology)

plt.xscale('symlog', linthresh=1)
plt.xlim(1, max_val1 * 1.05)
plt.title("Cumulative Distribution: Gene Set Sizes (All)", fontsize=16)
plt.xlabel("Gene Set Size (symlog scale)", fontsize=14)
plt.ylabel("Cumulative Proportion", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("plots/cumulative_distribution_gene_set_sizes_all_cdf_symlog.png")
plt.show()

# -------------------------------
# 2. Cumulative Distribution: Gene Set Sizes (minSize to maxSize) (linear scale)
# -------------------------------
cat2 = "Distribution of Gene Set Sizes (minSize to maxSize):"
plt.figure(figsize=(10, 7))
max_val2 = 0
for ontology in ontologies:
    if cat2 in all_distributions[ontology]:
        data = np.array(all_distributions[ontology][cat2], dtype=int)
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        max_val2 = max(max_val2, sorted_data[-1])
        plt.step(sorted_data, cdf, where="post", label=ontology)

plt.xlim(0, max_val2 * 1.05)
plt.title("Cumulative Distribution: Gene Set Sizes (minSize to maxSize)", fontsize=16)
plt.xlabel("Gene Set Size", fontsize=14)
plt.ylabel("Cumulative Proportion", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("plots/cumulative_distribution_gene_set_sizes_minSize_to_maxSize.png")
plt.show()

# -------------------------------
# 3. Distribution: Leaf Path Lengths from Root (Bar Plot)
# -------------------------------
cat3 = "Distribution of Leaf Path Lengths from Root:"

# Gather the union of unique path lengths and frequency counts per ontology.
unique_vals_set = set()
ontology_freq = {}  # key: ontology, value: dict mapping path length -> count
for ontology in ontologies:
    if cat3 in all_distributions[ontology]:
        data = np.array(all_distributions[ontology][cat3], dtype=int)
        vals, counts = np.unique(data, return_counts=True)
        freq_dict = dict(zip(vals, counts))
        ontology_freq[ontology] = freq_dict
        unique_vals_set.update(vals)

unique_vals = np.array(sorted(unique_vals_set))
x = np.arange(len(unique_vals))
width = 0.25  # width for each bar group
offsets = np.linspace(-width, width, num=len(ontologies))

fig, ax = plt.subplots(figsize=(10, 7))
for i, ontology in enumerate(ontologies):
    counts = [ontology_freq.get(ontology, {}).get(val, 0) for val in unique_vals]
    ax.bar(x + offsets[i], counts, width=width, label=ontology)

ax.set_xticks(x)
ax.set_xticklabels(unique_vals)
ax.set_xlabel("Path Length", fontsize=14)
ax.set_ylabel("Frequency", fontsize=14)
ax.set_title("Distribution: Leaf Path Lengths from Root", fontsize=16)
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("plots/distribution_leaf_path_lengths_bar.png")
plt.show()

# -------------------------------
# Cumulative Distribution: Leaf Path Lengths from Root (Line Plot)
# -------------------------------
plt.figure(figsize=(10, 7))
for ontology in ontologies:
    if cat3 in all_distributions[ontology]:
        data = np.array(all_distributions[ontology][cat3], dtype=int)
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        plt.step(sorted_data, cdf, where="post", label=ontology)

if len(sorted_data) > 0:
    plt.xlim(sorted_data[0], sorted_data[-1] + 1)
plt.title("Cumulative Distribution: Leaf Path Lengths from Root", fontsize=16)
plt.xlabel("Path Length", fontsize=14)
plt.ylabel("Cumulative Proportion", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("plots/cumulative_distribution_leaf_path_lengths.png")
plt.show()

# -------------------------------
# Violin Plot: Leaf Path Lengths from Root
# -------------------------------
data_list = []
for ontology in ontologies:
    if cat3 in all_distributions[ontology]:
        for val in all_distributions[ontology][cat3]:
            data_list.append({'Ontology': ontology, 'Path Length': int(val)})
df = pd.DataFrame(data_list)

plt.figure(figsize=(10, 7))
sns.violinplot(x="Ontology", y="Path Length", data=df, inner="point", cut=0)
plt.title("Violin Plot: Leaf Path Lengths from Root", fontsize=16)
plt.xlabel("Ontology", fontsize=14)
plt.ylabel("Path Length", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("plots/violin_plot_leaf_path_lengths.png")
plt.show()

# -------------------------------
# 4. Cumulative Distribution: Set Differences (Parent - Child) (linear scale)
# -------------------------------
cat4 = "Distribution of Set Differences (Parent - Child):"
plt.figure(figsize=(10, 7))
max_val4 = 0
for ontology in ontologies:
    if cat4 in all_distributions[ontology]:
        data = np.array(all_distributions[ontology][cat4], dtype=int)
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        max_val4 = max(max_val4, sorted_data[-1])
        plt.step(sorted_data, cdf, where="post", label=ontology)

plt.xlim(0, max_val4 * 1.05)
plt.title("Cumulative Distribution: Set Differences (Parent - Child)", fontsize=16)
plt.xlabel("Set Difference (Parent - Child)", fontsize=14)
plt.ylabel("Cumulative Proportion", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("plots/cumulative_distribution_set_differences.png")
plt.show()